In [1]:
"""
Gradient boosting from scratch, two ways, to make the XGBoost paper's core
contribution concrete: it's one equation.

Both models boost on the logistic loss for binary classification:
  L(y, p) = -[y*log(p) + (1-y)*log(1-p)],  p = sigmoid(raw_score)
  gradient g = p - y
  hessian  h = p * (1 - p)

VANILLA (first-order, classic Friedman-style GBM):
  Fit a regression tree to the negative gradient (-g), splitting on plain
  variance reduction (MSE), same as fitting any regression target.
  Leaf value = mean(-g) in that leaf. No second-order information used at
  all -- this is exactly gradient descent in function space.

XGBOOST-STYLE (second-order):
  Use the SAME boosting loop, but:
    1. split selection uses the exact-greedy gain formula derived from the
       second-order Taylor expansion of the loss:
         gain = 0.5 * [ GL^2/(HL+lam) + GR^2/(HR+lam) - G^2/(H+lam) ] - gamma
    2. leaf value uses the closed-form optimum:
         w* = -G / (H + lam)
  This is the one mathematical difference the post claims it is.
"""

import numpy as np
from sklearn.tree import DecisionTreeRegressor


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


# -vanilla --
class VanillaGBM:
    """First-order gradient boosting: fit a plain regression tree to -gradient."""

    def __init__(self, n_estimators=50, max_depth=3, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lr = learning_rate
        self.trees = []
        self.base_score = 0.0

    def fit(self, X, y):
        p = np.full(len(y), y.mean())
        self.base_score = float(np.log(p[0] / (1 - p[0])))  # logit of base rate
        raw = np.full(len(y), self.base_score)

        for _ in range(self.n_estimators):
            p = sigmoid(raw)
            neg_grad = y - p                       # -g, g = p - y
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, neg_grad)                  # plain MSE split, no Hessian
            update = tree.predict(X)
            raw += self.lr * update
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        raw = np.full(len(X), self.base_score)
        for tree in self.trees:
            raw += self.lr * tree.predict(X)
        return sigmoid(raw)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


# xgboost-style --
class _XGBNode:
    __slots__ = ("feature", "threshold", "left", "right", "weight", "is_leaf")


class _XGBTree:
    """Exact-greedy regression tree split on the second-order gain formula."""

    def __init__(self, max_depth, lam, gamma, min_child_weight):
        self.max_depth = max_depth
        self.lam = lam
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.root = None

    def _leaf_weight(self, g, h):
        return -g.sum() / (h.sum() + self.lam)

    def _best_split(self, X, g, h):
        best_gain, best_feat, best_thr = 0.0, None, None
        G, H = g.sum(), h.sum()
        n_features = X.shape[1]
        for f in range(n_features):
            order = np.argsort(X[:, f])
            xf, gf, hf = X[order, f], g[order], h[order]
            GL = np.cumsum(gf)[:-1]
            HL = np.cumsum(hf)[:-1]
            GR = G - GL
            HR = H - HL
            valid = (HL >= self.min_child_weight) & (HR >= self.min_child_weight)
            if not np.any(valid):
                continue
            gain = 0.5 * (GL**2 / (HL + self.lam) + GR**2 / (HR + self.lam)
                           - G**2 / (H + self.lam)) - self.gamma
            gain = np.where(valid, gain, -np.inf)
            # only split between distinct consecutive x-values
            distinct = xf[1:] != xf[:-1]
            gain = np.where(distinct, gain, -np.inf)
            i = np.argmax(gain)
            if gain[i] > best_gain:
                best_gain, best_feat = gain[i], f
                best_thr = (xf[i] + xf[i + 1]) / 2.0
        return best_feat, best_thr, best_gain

    def _build(self, X, g, h, depth):
        node = _XGBNode()
        if depth >= self.max_depth or len(g) < 2 * self.min_child_weight:
            node.is_leaf = True
            node.weight = self._leaf_weight(g, h)
            return node

        feat, thr, gain = self._best_split(X, g, h)
        if feat is None or gain <= 0:
            node.is_leaf = True
            node.weight = self._leaf_weight(g, h)
            return node

        mask = X[:, feat] <= thr
        node.is_leaf = False
        node.feature, node.threshold = feat, thr
        node.left = self._build(X[mask], g[mask], h[mask], depth + 1)
        node.right = self._build(X[~mask], g[~mask], h[~mask], depth + 1)
        return node

    def fit(self, X, g, h):
        self.root = self._build(X, g, h, depth=0)
        return self

    def _predict_one(self, x, node):
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.weight

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])


class XGBScratch:
    """Second-order gradient boosting -- the actual XGBoost split/leaf rule."""

    def __init__(self, n_estimators=50, max_depth=3, learning_rate=0.1,
                 lam=1.0, gamma=0.0, min_child_weight=1.0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lr = learning_rate
        self.lam = lam
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.trees = []
        self.base_score = 0.0

    def fit(self, X, y):
        p0 = y.mean()
        self.base_score = float(np.log(p0 / (1 - p0)))
        raw = np.full(len(y), self.base_score)

        for _ in range(self.n_estimators):
            p = sigmoid(raw)
            g = p - y                      # gradient of logloss
            h = p * (1 - p)                # hessian of logloss
            tree = _XGBTree(self.max_depth, self.lam, self.gamma, self.min_child_weight)
            tree.fit(X, g, h)
            raw += self.lr * tree.predict(X)
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        raw = np.full(len(X), self.base_score)
        for tree in self.trees:
            raw += self.lr * tree.predict(X)
        return sigmoid(raw)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)
